# 03. Random split: extra seeds of the full-data models

Trains the full-data models again with seeds 43 and 44 and scores them on the test set. Part A covers the ProtT5-XL arm and Part B the structure arm.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [2]:
import os
from glob import glob

DRIVE_ROOT = "/content/drive/MyDrive/Diplomska_v2"
if not os.path.isdir(DRIVE_ROOT):
    shared = glob("/content/drive/.shortcut-targets-by-id/*/Diplomska_v2")
    if shared:
        DRIVE_ROOT = shared[0]
assert os.path.isdir(DRIVE_ROOT), f"Folder not found: {DRIVE_ROOT}"

PDB_SET = "kinase_domain"
PLM_MODEL = "prot_t5_xl_uniref50"
STOP_AFTER_FOLDS = None

os.environ["DTA_DATA_ROOT"] = DRIVE_ROOT
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("DTA_DATA_ROOT:", os.environ["DTA_DATA_ROOT"])
print("Structure set:", PDB_SET, "| language model:", PLM_MODEL)

DTA_DATA_ROOT: /content/drive/MyDrive/Diplomska_v2
Structure set: kinase_domain | language model: prot_t5_xl_uniref50


## 3. GPU

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Torch version: 2.11.0+cu130
CUDA available: True
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 4. Install packages
`esm` is the EvolutionaryScale package that provides ESM C (not `fair-esm`). `biopython` is needed because the embedding script reads the PDB files.

In [6]:
!pip install -q torch_geometric biopython esm

## 5. Go to the folder

In [7]:
%cd {DRIVE_ROOT}
!pwd

/content/drive/MyDrive/Diplomska_v2
/content/drive/MyDrive/Diplomska_v2


## 6. Check the uploaded files

In [8]:
import argparse, json
import pandas as pd

required = [
    "Structure_Sets.py", "Paths.py", "RunTag.py", "Conditions.py", "Protein_Embeddings.py",
    "compute_protein_embeddings.py", "Protein_Representation.py", "Creating_Train_and_Test_set.py",
    "GNNNet.py", "PLMNet.py", "DTADataset.py", "Checkpointing.py", "Seeding.py", "Emetrics.py",
    "training_5_folds.py", "testing.py", "report_best_epochs.py",
]
missing = [f for f in required if not os.path.exists(f)]
print("Missing code files:", missing or "none")

from RunTag import add_run_args, run_tag
from Seeding import SEED
parser = argparse.ArgumentParser()
add_run_args(parser)
print("--pdb-set flag accepts", PDB_SET + ":", PDB_SET in parser.format_help())

from Paths import paths_for
PATHS = paths_for("davis", PDB_SET)
assert PATHS.root == DRIVE_ROOT, f"Data root {PATHS.root} is not {DRIVE_ROOT}"
for label, path in [("pair table", PATHS.csv), ("train folds", PATHS.train_folds), ("test fold", PATHS.test_fold)]:
    print(f"{label}: {os.path.exists(path)}  {path}")

names = pd.read_csv(PATHS.csv)["protein"].unique()
pdb_missing = [n for n in names if not os.path.exists(PATHS.pdb_file(n))]
print(f"proteins in pair table: {len(names)} (expected 442)")
print(f"PDB files missing for {len(pdb_missing)} proteins", pdb_missing[:3] or "", "<- expected 0")
print("PDB folder:", PATHS.pdb)

graph_files = glob(os.path.join(PATHS.protein_graph_gml, "*_contact_map_graph.gml"))
graph_names = {os.path.basename(f)[: -len("_contact_map_graph.gml")] for f in graph_files}
print(f"protein graphs: {len(graph_files)} (expected 442), cover all proteins: {set(names) <= graph_names}")
print("protein graph folder:", PATHS.protein_graph_gml)

ligand_files = glob(os.path.join(PATHS.ligand_graphs, "*.gml"))
print(f"ligand graphs: {len(ligand_files)} (expected 68)")

with open(PATHS.train_folds) as f:
    folds = json.load(f)
print("fold sizes:", [len(x) for x in folds], "(expected 5010, 5009, 5009, 5009, 5009)")

TAG_PLM = run_tag("plm", PLM_MODEL, SEED, False, PDB_SET)
TAG_STRUCTURE = run_tag("structure", None, SEED, False, PDB_SET)
print("Run tags:", TAG_PLM, "|", TAG_STRUCTURE, "(both must end with pdb2)")
assert TAG_PLM.endswith("pdb2") and TAG_STRUCTURE.endswith("pdb2")

stale = [p for p in glob(os.path.join(PATHS.models, "*")) + glob(os.path.join(PATHS.results, "*"))
         if "pdb2" not in os.path.basename(p)]
print("Files in models/results without the pdb2 tag:", stale or "none")

Missing code files: none
--pdb-set flag accepts kinase_domain: True
pair table: True  /content/drive/MyDrive/Diplomska_v2/data/davis/davis.csv
train folds: True  /content/drive/MyDrive/Diplomska_v2/data/davis/train_folds.txt
test fold: True  /content/drive/MyDrive/Diplomska_v2/data/davis/test_fold.txt
proteins in pair table: 442 (expected 442)
PDB files missing for 0 proteins  <- expected 0
PDB folder: /content/drive/MyDrive/Diplomska_v2/data/davis/pdb_kinase_domain
protein graphs: 442 (expected 442), cover all proteins: True
protein graph folder: /content/drive/MyDrive/Diplomska_v2/data/davis/protein_graphs_kinase_domain/gml
ligand graphs: 68 (expected 68)
fold sizes: [5010, 5009, 5009, 5009, 5009] (expected 5010, 5009, 5009, 5009, 5009)
Run tags: prot_t5_xl_uniref50_pdb2 | pdb2 (both must end with pdb2)
Files in models/results without the pdb2 tag: none


## 7. Helpers

In [9]:
import re, statistics, subprocess

QUIET_PREFIXES = ("Train epoch:", "Training on ", "Predicting for validation data",
                  "Make prediction for", "MSE improved at epoch", "No improvement since epoch")
VALIDATION_LINE = re.compile(r"^Epoch (\d+) - Validation MSE")

def run(args, stop_marker=None, every=10):
    # Prints everything except per-epoch noise; a validation line is shown every `every` epochs.
    cmd = ["python", "-u"] + args
    print("Running:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stopped = False
    try:
        for line in proc.stdout:
            match = VALIDATION_LINE.match(line)
            if match:
                if int(match.group(1)) % every == 0:
                    print(line, end="")
                continue
            if line.startswith(QUIET_PREFIXES):
                continue
            print(line, end="")
            if stop_marker and stop_marker in line:
                stopped = True
                print("\nStopping here as requested. Rerun this cell to continue; finished folds are skipped.")
                break
    finally:
        if proc.poll() is None:
            proc.terminate()
        proc.wait()
    if not stopped and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}, see the output above.")

def arm_flags(protein_repr):
    flags = ["--protein-repr", protein_repr, "--pdb-set", PDB_SET]
    if protein_repr == "plm":
        flags += ["--plm-model", PLM_MODEL]
    return flags

FLAGS_PLM = " ".join(arm_flags("plm"))
FLAGS_STRUCTURE = " ".join(arm_flags("structure"))

def stop_marker():
    return f"Training for fold {STOP_AFTER_FOLDS}..." if STOP_AFTER_FOLDS else None

def median_best_epoch(tag):
    path = os.path.join(PATHS.results, f"best_epochs_{tag}.json")
    with open(path) as f:
        records = json.load(f)
    folds = [str(i) for i in range(5)]
    absent = [i for i in folds if i not in records]
    assert not absent, f"Folds {absent} are not finished yet in {path}"
    print(f"{'fold':>4} {'best_epoch':>10} {'last_epoch':>10} {'best_val_mse':>13}")
    for i in folds:
        r = records[i]
        print(f"{i:>4} {r['best_epoch']:>10} {r['last_epoch']:>10} {r['best_val_mse']:>13.4f}")
    median = int(round(statistics.median(records[i]["best_epoch"] for i in folds)))
    print("Median best epoch (use for the full-data model):", median)
    return median

def show_result(model_name, tag, full=False):
    path = os.path.join(PATHS.results, f"result_{model_name}_davis_{tag}{'_full' if full else ''}.txt")
    print(path)
    with open(path) as f:
        print(f.read())

## 8. Seeds

In [11]:
SEEDS = [43, 44]

## 9. Seed helpers

In [12]:
def seed_tag(protein_repr, seed):
    plm = PLM_MODEL if protein_repr == "plm" else None
    tag = run_tag(protein_repr, plm, seed, False, PDB_SET)
    assert tag.endswith(f"s{seed}"), tag
    return tag

def train_and_test_full(protein_repr, model_name, seed, epochs):
    tag = seed_tag(protein_repr, seed)
    flags = arm_flags(protein_repr) + ["--seed", str(seed)]
    model_file = os.path.join(PATHS.models, f"model_{model_name}_davis_{tag}_full.model")
    result_file = os.path.join(PATHS.results, f"result_{model_name}_davis_{tag}_full.txt")
    print(f"=== {model_name} seed {seed}, tag {tag}, {epochs} epochs ===")
    if os.path.exists(model_file):
        print("Model already finished, skipping training:", model_file)
    else:
        run(["training_5_folds.py"] + flags + ["--mode", "full", "--epochs", str(epochs)], every=50)
    if os.path.exists(result_file):
        print("Result already exists, skipping test:", result_file)
    else:
        run(["testing.py"] + flags + ["--full"])
    show_result(model_name, tag, full=True)

def metric_across_seeds(model_name, protein_repr, metric, seeds):
    values = []
    for seed in seeds:
        tag = run_tag(protein_repr, PLM_MODEL if protein_repr == "plm" else None, seed, False, PDB_SET)
        path = os.path.join(PATHS.results, f"result_{model_name}_davis_{tag}_full.txt")
        value = None
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    if line.startswith(metric + " ") and "mean:" in line:
                        value = float(line.split("mean:")[1].split()[0])
        values.append(value)
    return values

def seed_table(model_name, protein_repr, seeds):
    print(f"{model_name}, full data")
    print(f"{'seed':>6} {'MSE':>8} {'CI':>8}")
    mses = metric_across_seeds(model_name, protein_repr, "mse", seeds)
    cis = metric_across_seeds(model_name, protein_repr, "ci", seeds)
    for seed, mse, ci in zip(seeds, mses, cis):
        show = lambda v: f"{v:8.4f}" if v is not None else f"{'-':>8}"
        print(f"{seed:>6} {show(mse)} {show(ci)}")
    done = [(m, c) for m, c in zip(mses, cis) if m is not None and c is not None]
    if len(done) >= 2:
        for name, vals in (("MSE", [m for m, _ in done]), ("CI", [c for _, c in done])):
            print(f"{name}: mean {statistics.mean(vals):.4f}  std {statistics.stdev(vals):.4f}  (n={len(vals)})")
    else:
        print("Need at least two finished seeds for a mean and std.")

# Part A: ProtT5-XL, full data, extra seeds

In [ ]:
FULL_EPOCHS_PLM = median_best_epoch(TAG_PLM)
for seed in SEEDS:
    train_and_test_full("plm", "PLMNet", seed, FULL_EPOCHS_PLM)

fold best_epoch last_epoch  best_val_mse
   0        800        900        0.1739
   1        694        794        0.1705
   2        556        656        0.1873
   3        430        530        0.1719
   4        734        834        0.1694
Median best epoch (use for the full-data model): 694
=== PLMNet seed 43, tag prot_t5_xl_uniref50_pdb2_s43, 694 epochs ===
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --plm-model prot_t5_xl_uniref50 --seed 43 --mode full --epochs 694
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  prot_t5_xl_uniref50
Standardized embeddings:  False
Seed:  43
Learning rate:  0.001
Epochs:  694
Data root:  /content/drive/MyDrive/Diplomska_v2
Using device: cuda
Run tag:  prot_t5_xl_uniref50_pdb2_s43
Loaded prot_t5_xl_uniref50 embeddings for 442 proteins, dim 1024
/content/drive/MyDrive/Diplomska_v2/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecate

## A summary: ProtT5-XL across seeds

In [ ]:
seed_table("PLMNet", "plm", [42] + SEEDS)

PLMNet, full data
  seed      MSE       CI
    42   0.1666   0.9110
    43   0.1789   0.9093
    44   0.1643   0.9095
MSE: mean 0.1699  std 0.0079  (n=3)
CI: mean 0.9099  std 0.0009  (n=3)


# Part B: structure baseline, full data, extra seeds

In [ ]:
FULL_EPOCHS_STRUCTURE = median_best_epoch(TAG_STRUCTURE)
for seed in SEEDS:
    train_and_test_full("structure", "GNNNet", seed, FULL_EPOCHS_STRUCTURE)

fold best_epoch last_epoch  best_val_mse
   0       1020       1120        0.1857
   1        692        792        0.1821
   2        982       1082        0.1873
   3        742        842        0.1754
   4        804        904        0.1779
Median best epoch (use for the full-data model): 804
=== GNNNet seed 43, tag pdb2_s43, 804 epochs ===
Running: python -u training_5_folds.py --protein-repr structure --pdb-set kinase_domain --seed 43 --mode full --epochs 804
Dataset:  davis
Mode:  full
Protein representation:  structure
Structure set:  kinase_domain
Seed:  43
Learning rate:  0.001
Epochs:  804
Data root:  /content/drive/MyDrive/Diplomska_v2
Using device: cuda
Run tag:  pdb2_s43
GNNNet Loaded
/content/drive/MyDrive/Diplomska_v2/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
Saved full-data model after 804 epochs to /content/drive/MyDrive/Diplomska_v2/models_davis/model_GNNNet_davis_pdb2_s43_full.model
Ru

## B summary: structure across seeds

In [13]:
seed_table("GNNNet", "structure", [42] + SEEDS)

GNNNet, full data
  seed      MSE       CI
    42   0.1653   0.9104
    43   0.1728   0.9140
    44   0.1693   0.9100
MSE: mean 0.1691  std 0.0038  (n=3)
CI: mean 0.9115  std 0.0022  (n=3)


In [14]:
for m in ["esmc_300m", "esmc_600m", "prot_t5_xl_uniref50"]:
    PLM_MODEL = m
    print(m)
    seed_table("PLMNet", "plm", [42] + SEEDS)
    print()
seed_table("GNNNet", "structure", [42] + SEEDS)

esmc_300m
PLMNet, full data
  seed      MSE       CI
    42   0.1625   0.9116
    43   0.1711   0.9119
    44   0.1625   0.9064
MSE: mean 0.1654  std 0.0050  (n=3)
CI: mean 0.9100  std 0.0031  (n=3)

esmc_600m
PLMNet, full data
  seed      MSE       CI
    42   0.1586   0.9075
    43   0.1676   0.9126
    44   0.1608   0.9128
MSE: mean 0.1623  std 0.0047  (n=3)
CI: mean 0.9110  std 0.0030  (n=3)

prot_t5_xl_uniref50
PLMNet, full data
  seed      MSE       CI
    42   0.1666   0.9110
    43   0.1789   0.9093
    44   0.1643   0.9095
MSE: mean 0.1699  std 0.0079  (n=3)
CI: mean 0.9099  std 0.0009  (n=3)

GNNNet, full data
  seed      MSE       CI
    42   0.1653   0.9104
    43   0.1728   0.9140
    44   0.1693   0.9100
MSE: mean 0.1691  std 0.0038  (n=3)
CI: mean 0.9115  std 0.0022  (n=3)
